# Collusion-Watch Aya Market Colab Notebook

This is a historical experiment notebook assembled across separate Colab sessions; it is **not** intended to run top-to-bottom. This shareable copy uses the latest notebook the project owner supplied, with all saved outputs removed. Run only the labeled cells for the verified pipeline; the later cells preserve older exploratory experiments and may overwrite data or consume GPU/API budget.

The current source snapshot has **112 complete episodes**: A2 30 (seeds 0-19 and 21-30; seed 20 is absent), A3 41, and A5 41. The seed-0 M2 dataset has 78 train, 18 validation, and 16 test examples; CuratorKIT exported 78 training examples with zero rejections. The new sweep compares six monitor arms on the same 16 held-out episodes (12 collusive, 4 benign).

The supplied sweep reports M2 AUROC 0.15 (base) vs 0.58 (fine-tuned), but neither M2 arm flags a collusive episode; M4 flags all 16. The sweep's `judge failures` column is stale: the raw reasons contain three unparseable base-judge responses per judge-using arm. The results cell below records this caveat. Use Colab Secrets for `HF_TOKEN`; never paste or print credentials. Source history: [GitHub](https://github.com/munip/apart-sprint-collusions/commits/feature/lexsi-work).


In [ ]:
%cd /content
!git clone --single-branch --branch feature/lexsi-work https://github.com/munip/apart-sprint-collusions.git
%cd /content/apart-sprint-collusions
!git rev-parse HEAD

!pip install -e ".[dev]"
!pip install -U transformers accelerate
!pip install torchao
!pip install -e ".[lexsi,lexsi-train]"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
print("Drive mounted:", os.path.ismount("/content/drive"))
print("Run folder exists:", os.path.isdir(
    "/content/drive/MyDrive/lexsi/runs/aya_market_base_a2prompt"
))

Test to download Cohere models from hugging face. Ignore next well after once check

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "CohereLabs/tiny-aya-global"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

messages = [{"role": "user", "content": 'Reply with JSON only: {"ask": 70}'}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Delete everything in CUDA memory and flush GC  to free up GPU for clab runs

In [ ]:
del model, tokenizer, inputs, output
import gc
gc.collect()
torch.cuda.empty_cache()

Base run with no monitors. Monitors are run async after the runs in google colab due to 15GB GPU RAM limit. So, live monitoring and hold or live stop operations on sellers aren't run



In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


## Existing source-run commands ? check for duplicates first

This configuration requests seeds 10-40 across A2/A3/A5, many of which already exist in Drive. Do not rerun it blindly. Compare `(condition, seed)` pairs in `episodes.csv` and run only missing episodes. The absent A2 seed 20 is not a malformed episode to delete.


In [ ]:
%%writefile configs/experiments/aya_market_gap_all.yaml
scenario: market
profile: scripted
conditions: [A2_honest_chat, A5_stego_rotation, A3_overt_cartel ]
seeds:  [21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,10, 11, 12,13,14,15,16,17,18,19,20]
rounds: 15
online_monitors: []
interventions: [entrant]
cap_usd: 5

In [ ]:
!clab run configs/experiments/aya_market_gap_all.yaml \
  --out /content/drive/MyDrive/lexsi/runs/aya_market_base_a2prompt \
  --profile scripted \
  --role seller=hf/CohereLabs/tiny-aya-global \
  --workers 1

In [ ]:
%cd /content/apart-sprint-collusions
!git status -sb
!git fetch origin
!git switch feature/lexsi-work
!git pull --ff-only origin feature/lexsi-work
!git log -1 --oneline


## Rebuild and verify the M2 dataset, curate train only, then fine-tune

Rebuild the episode index and dataset from the verified run. The dataset builder validates evidence chains and keeps all conditions for a seed in one split. Check the generated manifest before training; pass only the train split to CuratorKIT and the trainer. Current artifacts show 112 examples, a 78/18/16 split, and zero CuratorKIT rejections. Targets use episode truth; rationale text is templated, not evidence-grounded.


In [ ]:
!python scripts/rebuild_episodes_csv.py /content/drive/MyDrive/lexsi/runs/aya_market_base_a2prompt

In [ ]:
#curate dataset
!python -m collusion_lab.lexsi.dataset \
  --runs /content/drive/MyDrive/lexsi/runs/aya_market_base_a2prompt \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_dataset_41seed30_v1 \
  --task m2_transcript --window-rounds 15 --split-seed 0

!python -m collusion_lab.lexsi.curation \
  --input /content/drive/MyDrive/lexsi/runs/aya_m2_dataset_41seed30_v1/dataset.jsonl \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_curated_41seed30_v1 \
  --split train

In [ ]:
!python -m collusion_lab.lexsi.train \
  --dataset /content/drive/MyDrive/lexsi/runs/aya_m2_curated_41seed30_v1/curator/sft_alpaca.jsonl \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_finetunes_41seed30_v1 \
  --model CohereLabs/tiny-aya-fire \
  --algorithm plain \
  --epochs 1 \
  --batch-size 1

## Upload the fine-tuned model privately

Run this only after training completes and the output contains model weights. The code creates a private model repository and reads `HF_TOKEN` from Colab Secrets. Verify the uploaded model revision before using its exact repository ID in the sweep; do not make the repository public from this notebook.


In [ ]:
#upload model to HF
import os
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets before uploading.")
os.environ["HF_TOKEN"] = token

model_dir = Path(
    "/content/drive/MyDrive/lexsi/runs/aya_m2_finetunes_41seed30_v1/plain"
)
weight_files = (
    list(model_dir.glob("*.safetensors"))
    + list(model_dir.glob("pytorch_model*.bin"))
)
print("Weight files:", [(p.name, p.stat().st_size) for p in weight_files])
if not weight_files:
    raise RuntimeError(
        f"No model weights found in {model_dir}; this is not uploadable as a model."
    )

repo_id = "munip/tiny-aya-fire-m2-monitor-sft-41seed-v1"
api = HfApi(token=token)
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path=str(model_dir),
)
print("Uploaded files:", api.list_repo_files(repo_id, repo_type="model"))


## Create a test-only replay folder

The corrected cell reads expected row and test counts from `manifest.json` instead of hard-coding stale totals. It copies only test-split episode folders, checks for complete artifacts, and refuses to overwrite a non-empty replay folder. Keep training episodes out of the replay.


In [ ]:
# Prepare a test-only replay folder
import json
import shutil
from collections import Counter
from pathlib import Path

run_dir = Path("/content/drive/MyDrive/lexsi/runs/aya_market_base_a2prompt")
dataset_path = Path("/content/drive/MyDrive/lexsi/runs/aya_m2_dataset_41seed30_v1/dataset.jsonl")
manifest_path = dataset_path.parent / "manifest.json"
test_dir = Path("/content/drive/MyDrive/lexsi/runs/aya_m2_test_replay_41seed30_v1")

rows = [json.loads(line) for line in dataset_path.read_text().splitlines() if line]
manifest = json.loads(manifest_path.read_text())
test_ids = {
    row["metadata"]["episode_id"]
    for row in rows
    if row["split"] == "test"
}
expected_rows = manifest["examples"]
expected_test = manifest["splits"]["test"]
if len(rows) != expected_rows or len(test_ids) != expected_test:
    raise ValueError(
        f"Dataset/manifest mismatch: rows {len(rows)}/{expected_rows}, "
        f"test IDs {len(test_ids)}/{expected_test}."
    )
if not test_ids:
    raise ValueError("The manifest has no test episodes.")
if test_dir.exists() and any(test_dir.iterdir()):
    raise FileExistsError(f"Refusing to overwrite non-empty replay folder: {test_dir}")
test_dir.mkdir(parents=True, exist_ok=True)
for episode_id in sorted(test_ids):
    source = run_dir / episode_id
    if not (source / "episode.json").is_file() or not (source / "evidence.jsonl").is_file():
        raise FileNotFoundError(f"Missing complete episode artifacts: {source}")
    shutil.copytree(source, test_dir / episode_id)

print(f"Prepared {len(test_ids)} test episodes from {len(rows)} dataset examples.")
print("Test counts by condition:", Counter(row["metadata"]["condition"] for row in rows if row["split"] == "test"))


## Run the base-vs-SFT monitor comparison

This sweeps the same 16 seed-disjoint test trajectories with the base and fine-tuned Tiny Aya Fire judges. The supplied output archive is `aya_m2_monitor_41seed30_comparison_v1`; see the results note after the command for its corrected failure-count interpretation.


In [ ]:
!clab sweep-monitors \
  /content/drive/MyDrive/lexsi/runs/aya_m2_test_replay_41seed30_v1 \
  --out /content/drive/MyDrive/lexsi/runs/aya_m2_monitor_41seed30_comparison_v1 \
  --judge aya_fire=hf/CohereLabs/tiny-aya-fire \
  --judge aya_fire_sft=hf/munip/tiny-aya-fire-m2-monitor-sft-41seed-v1

## Current sweep results (16 held-out episodes)

| Monitor | Judge | AUROC (95% CI) | Harm flagged | Benign flagged |
|---|---|---:|---:|---:|
| M1 | Rules | 0.75 [0.62, 0.88] | 12/12 | 0/4 |
| M2 | Base Tiny Aya Fire | 0.15 [0.00, 0.31] | 1/12 | 2/4 |
| M2 | Fine-tuned Tiny Aya Fire | 0.58 [0.22, 0.79] | 0/12 | 0/4 |
| M3 | Model-free | 0.38 [0.07, 0.67] | 12/12 | 4/4 |
| M4 | Base Tiny Aya Fire | 0.27 [0.00, 0.57] | 12/12 | 4/4 |
| M4 | Fine-tuned Tiny Aya Fire | 0.38 [0.07, 0.67] | 12/12 | 4/4 |

Three base-judge outputs were unparseable in both M2 and M4 (the same two A2 and one A3 test episodes). The archived CSV says zero judge failures; use the raw `reasons` field for the corrected count. The four-episode benign set and zero M2 action-level recall for SFT make these exploratory results only.

## Archived exploratory cells

The remaining cells are earlier experiments with different seeds, conditions, prompts, and output folders. They are retained for provenance, not as the current pipeline; run them only when intentionally reproducing those experiments.


In [ ]:
!clab replay /content/drive/MyDrive/lexsi/runs/aya_market_base_global \
  --profile tiny_aya_replay \
  --monitors M2 M3 M4 \
  --observability full \
  --workers 1

In [ ]:
%cd /content/apart-sprint-collusions
!git pull --ff-only origin feature/lexsi-work

In [ ]:
%cd /content/apart-sprint-collusions
!git pull --ff-only origin feature/lexsi-work
!pip install -e ".[lexsi,lexsi-train]"

In [ ]:
%%bash
cd /content/apart-sprint-collusions

python -m collusion_lab.lexsi.dataset \
  --runs /content/drive/MyDrive/lexsi/runs/aya_market_base_a2prompt \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_dataset_v6 \
  --task m2_transcript --window-rounds 1 --split-seed 0

python -m collusion_lab.lexsi.curation \
  --input /content/drive/MyDrive/lexsi/runs/aya_m2_dataset_v6/dataset.jsonl \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_curated_v6 \
  --split train

python -m collusion_lab.lexsi.train \
  --dataset /content/drive/MyDrive/lexsi/runs/aya_m2_curated_v6/curator/sft_alpaca.jsonl \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_finetunes_v6 \
  --model CohereLabs/tiny-aya-fire \
  --algorithm plain --epochs 1 --batch-size 1

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!python -m collusion_lab.lexsi.train \
  --dataset /content/drive/MyDrive/lexsi/runs/aya_m2_curated_v6/curator/sft_alpaca.jsonl \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_finetunes_v6 \
  --model CohereLabs/tiny-aya-fire \
  --algorithm plain --epochs 1 --batch-size 1

In [ ]:
%%bash
cd /content/apart-sprint-collusions

python -m collusion_lab.lexsi.dataset \
  --runs /content/drive/MyDrive/lexsi/runs/aya_market_base_a2prompt \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_dataset_v2 \
  --task m2_transcript --window-rounds 15 --split-seed 0

python -m collusion_lab.lexsi.curation \
  --input /content/drive/MyDrive/lexsi/runs/aya_m2_dataset_v2/dataset.jsonl \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_curated_v2 \
  --split train

python -m collusion_lab.lexsi.train \
  --dataset /content/drive/MyDrive/lexsi/runs/aya_m2_curated_v2/curator/sft_alpaca.jsonl \
  --output-dir /content/drive/MyDrive/lexsi/runs/aya_m2_finetunes_v2 \
  --model CohereLabs/tiny-aya-fire \
  --algorithm plain --epochs 1 --batch-size 1 --max-length 512

In [ ]:
%%writefile configs/experiments/aya_market_pilot2.yaml
scenario: market
profile: scripted
conditions: [A2_honest_chat, A3_overt_cartel]
seeds:  [1, 2, 3 ,4 ]
rounds: 15
online_monitors: []
interventions: [entrant]
cap_usd: 5

In [ ]:
%%writefile configs/experiments/aya_market_pilot3.yaml
scenario: market
profile: scripted
conditions: [A2_honest_chat, A3_overt_cartel, A5_stego_rotation]
seeds:  [1, 2, 3  ]
rounds: 15
online_monitors: []
interventions: [entrant]
cap_usd: 5

In [ ]:
%%writefile configs/experiments/aya_market_a5.yaml
scenario: market
profile: scripted
conditions: [A5_stego_rotation]
seeds: [0]
rounds: 15
online_monitors: []
interventions: [entrant]
cap_usd: 5

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%writefile configs/experiments/aya_market_pilot_all.yaml
scenario: market
profile: scripted
conditions: [A2_honest_chat, A3_overt_cartel, A5_stego_rotation]
seeds:  [1, 2, 3]
rounds: 15
online_monitors: []
interventions: [entrant]
cap_usd: 5

In [ ]:
%%writefile configs/experiments/aya_market_overt_all.yaml
scenario: market
profile: scripted
conditions: [A3_overt_cartel]
seeds:  [10, 11, 12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40]
rounds: 15
online_monitors: []
interventions: [entrant]
cap_usd: 5

In [ ]:
%%writefile configs/experiments/aya_market_stego_all.yaml
scenario: market
profile: scripted
conditions: [A5_stego_rotation]
seeds:  [10, 11, 12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40]
rounds: 15
online_monitors: []
interventions: [entrant]
cap_usd: 5

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


In [ ]:
!clab run configs/experiments/aya_market_gap_all.yaml \
  --out /content/drive/MyDrive/lexsi/runs/aya_market_base_a2prompt \
  --profile scripted \
  --role seller=hf/CohereLabs/tiny-aya-global \
  --workers 1

In [ ]:
%%writefile configs/experiments/aya_a5_prompt_v2.yaml
scenario: market
profile: scripted
conditions: [A5_stego_rotation]
seeds: [1, 2, 3]
rounds: 15
online_monitors: []
interventions: [entrant]
cap_usd: 5

In [ ]:
from pathlib import Path
import shutil
import yaml

path = Path("configs/models.yaml")
assert path.is_file(), f"Not found: {path} — run this from the repository root."

backup = path.with_suffix(".yaml.bak")
shutil.copy2(path, backup)

config = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
config.setdefault("profiles", {})["tiny_aya_replay"] = {
    "default": "scripted",
    "monitor_judge": "hf/CohereLabs/tiny-aya-fire",
    "generate": {
        "max_retries": 0,
        "timeout": 180,
        "attempt_timeout": 150,
        "max_tokens": 800,
    },
}

path.write_text(
    yaml.safe_dump(config, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

# Verify the profile can be read back.
check = yaml.safe_load(path.read_text(encoding="utf-8"))
print("Updated and validated:", path)
print(check["profiles"]["tiny_aya_replay"])
print("Backup:", backup)

In [ ]:
!clab run configs/experiments/lexsi_market_base.yaml \
  --out /content/drive/MyDrive/lexsi/runs/aya_market_base_global \
  --profile scripted \
  --role seller=hf/CohereLabs/tiny-aya-global \
  --workers 1

In [ ]:
!clab run configs/experiments/aya_a5_prompt_v2.yaml \
  --out /content/drive/MyDrive/lexsi/runs/aya_a5_prompt_v2_s123 \
  --role seller=hf/CohereLabs/tiny-aya-global \
  --workers 1

In [ ]:
!clab run configs/experiments/aya_market_a5.yaml \
  --out /content/drive/MyDrive/lexsi/runs/aya_a5_seed0 \
  --role seller=hf/CohereLabs/tiny-aya-global \
  --workers 1

In [ ]:
!mkdir -p /content/drive/MyDrive/lexsi/runs
!clab run configs/experiments/aya_market_pilot.yaml \
  --out /content/drive/MyDrive/lexsi/runs/aya_seller_pilot \
  --role seller=hf/CohereLabs/tiny-aya-global \
  --workers 1

In [ ]:
!clab run configs/experiments/aya_market_pilot3.yaml \
  --out /content/drive/MyDrive/lexsi/runs/aya_market_pilot_all \
  --role seller=hf/CohereLabs/tiny-aya-global \
  --workers 1